In [1]:
%pip install -U sentence-transformers datasets torch ir_datasets transformers==4.41.2

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import DataLoader, Dataset
import ir_datasets
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
class MSMARCODataset(Dataset):
    def __init__(self, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        dataset = ir_datasets.load("msmarco-passage/train/triples-small")
        
        queries = {q.query_id: q.text for q in tqdm(dataset.queries_iter(), desc="Loading queries")}
        docs = {d.doc_id: d.text for d in tqdm(dataset.docs_iter(), desc="Loading docs")}
        
        self.data = []
        for i, item in enumerate(tqdm(dataset.docpairs_iter(), desc="Loading pairs")):
            self.data.append({
                'query': queries[item.query_id],
                'pos_doc': docs[item.doc_id_a],
                'neg_doc': docs[item.doc_id_b]
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        query = self.tokenizer(item['query'], truncation=True, padding='max_length', 
                               max_length=self.max_length, return_tensors='pt')
        pos_doc = self.tokenizer(item['pos_doc'], truncation=True, padding='max_length',
                                 max_length=self.max_length, return_tensors='pt')
        neg_doc = self.tokenizer(item['neg_doc'], truncation=True, padding='max_length',
                                 max_length=self.max_length, return_tensors='pt')
        return {
            'q_ids': query['input_ids'].squeeze(),
            'q_mask': query['attention_mask'].squeeze(),
            'p_ids': pos_doc['input_ids'].squeeze(),
            'p_mask': pos_doc['attention_mask'].squeeze(),
            'n_ids': neg_doc['input_ids'].squeeze(),
            'n_mask': neg_doc['attention_mask'].squeeze()
        }

    
dataset = MSMARCODataset(tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


Loading queries: 808731it [00:01, 410516.04it/s]
Loading docs: 8841823it [00:48, 182787.67it/s]
Loading pairs: 39780811it [02:37, 252588.33it/s]


In [4]:
print(tokenizer.decode(dataset[0]['q_ids'], skip_special_tokens=True))
print("positive:", tokenizer.decode(dataset[0]['p_ids'], skip_special_tokens=True))
print("negative:", tokenizer.decode(dataset[0]['n_ids'], skip_special_tokens=True))

is a little caffeine ok during pregnancy
positive: we don ’ t know a lot about the effects of caffeine during pregnancy on you and your baby. so it ’ s best to limit the amount you get each day. if you ’ re pregnant, limit caffeine to 200 milligrams each day. this is about the amount in 1½ 8 - ounce cups of coffee or one 12 - ounce cup of coffee.
negative: it is generally safe for pregnant women to eat chocolate because studies have shown to prove certain benefits of eating chocolate during pregnancy. however, pregnant women should ensure their caffeine intake is below 200 mg per day.


In [ ]:
from transformers import AutoModelForMaskedLM
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class SPLADE(nn.Module):
    def __init__(self, model_name='distilbert-base-uncased'):
        super().__init__()
        self.bert_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert_mlm(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        relu_logits = F.relu(logits)
        relu_logits = relu_logits * attention_mask.unsqueeze(-1)
        pooled, _ = torch.max(relu_logits, dim=1)
        pooled = torch.clamp(pooled, max=10.0)
        return torch.log1p(pooled + 1e-8)

def compute_loss(q_reps, p_reps, n_reps, lambda_q=1e-3, lambda_d=1e-4, tau=0.1):
    batch_size = q_reps.size(0)
    all_docs = torch.cat([p_reps, n_reps], dim=0)
    
    q_reps = F.normalize(q_reps, p=2, dim=-1)
    all_docs = F.normalize(all_docs, p=2, dim=-1)
    
    scores = torch.matmul(q_reps, all_docs.T) / tau
    scores = torch.clamp(scores, min=-100, max=100)
    
    labels = torch.arange(batch_size, device=q_reps.device)
    ce_loss = F.cross_entropy(scores, labels)
    
    l1_q = lambda_q * torch.mean(torch.abs(q_reps))
    l1_d = lambda_d * torch.mean(torch.abs(all_docs))
    
    total_loss = ce_loss + l1_q + l1_d
    
    if torch.isnan(total_loss) or torch.isinf(total_loss):
        return torch.tensor(0.0, device=q_reps.device, requires_grad=True)
    
    return total_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SPLADE().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)
scaler = torch.cuda.amp.GradScaler()

model.train()
for epoch in range(3):
    total_loss = 0.0
    running = 0.0
    valid_steps = 0

    for i, batch in enumerate(tqdm(dataloader, desc=f'Epoch {epoch+1}')):
        optimizer.zero_grad(set_to_none=True)

        try:
            with torch.cuda.amp.autocast(dtype=torch.float16):
                q_reps = model(batch['q_ids'].to(device), batch['q_mask'].to(device))
                p_reps = model(batch['p_ids'].to(device), batch['p_mask'].to(device))
                n_reps = model(batch['n_ids'].to(device), batch['n_mask'].to(device))
                loss = compute_loss(q_reps, p_reps, n_reps)

            if not torch.isnan(loss) and not torch.isinf(loss):
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                
                loss_val = loss.item()
                total_loss += loss_val
                running += loss_val
                valid_steps += 1
            else:
                scaler.update()
                continue

        except RuntimeError as e:
            scaler.update()
            continue

        if (i + 1) % 500 == 0 and valid_steps > 0:
            tqdm.write(f"Step {i+1}: avg loss = {running/valid_steps:.4f}")
            running = 0.0
            valid_steps = 0
        if (i + 1) % 3000 == 0 and valid_steps > 0:
            torch.save(model.state_dict(), f"splade_checkpoint3_{i+1}.pt")

    if len(dataloader) > 0:
        tqdm.write(f"Epoch {epoch+1} avg loss = {total_loss / len(dataloader):.4f}")


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 9513123b-f448-4ea9-8a9b-d23d4aa270db)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
/tmp/ipykernel_5128/1809362927.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1:   0%|          | 0/1243151 [00:00<?, ?it/s]/tmp/ipykernel_5128/1809362927.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
Epoch 1:   0%|          | 501/1243151 [01:01<37:36:57,  9.18it/s]

Step 500: avg loss = 0.8054


Epoch 1:   0%|          | 1001/1243151 [01:55<37:40:26,  9.16it/s]

Step 1000: avg loss = 0.4579


Epoch 1:   0%|          | 1500/1243151 [02:50<37:53:23,  9.10it/s]

...

Epoch 1:   1%|          | 10000/1243151 [18:11<37:35:13,  9.11it/s]

Step 10000: avg loss = 0.2428


Epoch 1:   1%|          | 10500/1243151 [19:06<37:57:46,  9.02it/s]

Step 10500: avg loss = 0.2411


Epoch 1:   1%|          | 10765/1243151 [19:34<37:21:31,  9.16it/s]

In [9]:
torch.save(model.state_dict(), f"splade_checkpoint.pt")

In [10]:
import torch
import random
from transformers import AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model = SPLADEFreezed().to(device)
# model.load_state_dict(torch.load('splade_checkpoint2_8000.pt'))
model.eval()

test_queries = [
    "what is python programming",
    "how to lose weight",
    "best restaurants in paris",
    "covid vaccine side effects",
    "machine learning tutorial",
    "climate change causes",
    "how to cook pasta",
    "bitcoin price prediction",
    "yoga benefits health",
    "electric cars pros cons"
]

reverse_voc = {v: k for k, v in tokenizer.vocab.items()}

with torch.no_grad():
    for query in test_queries:
        tokens = tokenizer(query, return_tensors='pt', padding=True, truncation=True)
        q_rep = model(tokens['input_ids'].to(device), tokens['attention_mask'].to(device))

        top_indices = torch.topk(q_rep[0], k=20).indices
        top_weights = torch.topk(q_rep[0], k=20).values
        
        print(f"\nQuery: {query}")
        print("Top tokens:", [(tokenizer.decode([idx]), f"{weight:.2f}") for idx, weight in zip(top_indices, top_weights)])



Query: what is python programming
Top tokens: [('python', '2.31'), ('programming', '1.02'), ('software', '0.95'), ('language', '0.83'), ('algebra', '0.74'), ('java', '0.61'), ('languages', '0.56'), ('c', '0.46'), ('web', '0.40'), ('file', '0.36'), ('hardware', '0.30'), ('function', '0.29'), ('tool', '0.15'), ('data', '0.05'), ('[unused4]', '0.00'), ('[unused3]', '0.00'), ('[PAD]', '0.00'), ('[unused0]', '0.00'), ('[unused1]', '0.00'), ('[unused2]', '0.00')]

Query: how to lose weight
Top tokens: [('weight', '1.82'), ('exercise', '1.30'), ('loss', '1.19'), ('lose', '1.02'), ('change', '0.99'), ('washing', '0.96'), ('yoga', '0.88'), ('gain', '0.74'), ('diet', '0.72'), ('healing', '0.67'), ('eating', '0.63'), ('increase', '0.54'), ('cure', '0.47'), ('training', '0.47'), ('breathing', '0.41'), ('sleep', '0.39'), ('cleaning', '0.33'), ('sleeping', '0.30'), ('body', '0.29'), ('drink', '0.29')]

Query: best restaurants in paris
Top tokens: [('paris', '1.89'), ('restaurant', '1.74'), ('hotel'

In [7]:
import transformers, inspect
print(transformers.__version__)

4.41.2


In [10]:
from tqdm import tqdm
import numpy as np

# Соберём длины всех текстов (query, pos_doc, neg_doc)
dataset = ir_datasets.load("msmarco-passage/train/triples-small")

queries = {q.query_id: q.text for q in tqdm(dataset.queries_iter(), desc="Loading queries")}
docs = {d.doc_id: d.text for d in tqdm(dataset.docs_iter(), desc="Loading docs")}

lengths = []

for item in tqdm(dataset.docpairs_iter(), desc="Scanning lengths"):
    q_len = len(queries[item.query_id].split())
    p_len = len(docs[item.doc_id_a].split())
    n_len = len(docs[item.doc_id_b].split())
    lengths.extend([q_len, p_len, n_len])

lengths = np.array(lengths)

print("count:", len(lengths))
print("min:", lengths.min())
print("max:", lengths.max())
print("mean:", lengths.mean())
print("median:", np.median(lengths))
print("quantile 0.90:", np.quantile(lengths, 0.90))
print("quantile 0.95:", np.quantile(lengths, 0.95))
print("quantile 0.99:", np.quantile(lengths, 0.99))


Loading queries: 808731it [00:01, 407801.91it/s]
Loading docs: 8841823it [00:38, 228610.25it/s]
Scanning lengths: 39780811it [06:31, 101527.61it/s]


count: 119342433
min: 1
max: 362
mean: 40.76910974322101
median: 42.0
quantile 0.90: 85.0
quantile 0.95: 99.0
quantile 0.99: 124.0


In [21]:
from transformers import AutoModelForMaskedLM
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import time

class SPLADE(nn.Module):
    def __init__(self, model_name='distilbert-base-uncased'):
        super().__init__()
        self.bert_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert_mlm(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        relu_logits = F.relu(logits)
        relu_logits = relu_logits * attention_mask.unsqueeze(-1)
        pooled, _ = torch.max(relu_logits, dim=1)
        pooled = torch.clamp(pooled, max=10.0)
        return torch.log1p(pooled + 1e-8)

def compute_loss(q_reps, p_reps, n_reps, lambda_q=1e-3, lambda_d=1e-4, tau=0.1):
    batch_size = q_reps.size(0)
    all_docs = torch.cat([p_reps, n_reps], dim=0)
    
    q_reps = F.normalize(q_reps, p=2, dim=-1)
    all_docs = F.normalize(all_docs, p=2, dim=-1)
    
    scores = torch.matmul(q_reps, all_docs.T) / tau
    scores = torch.clamp(scores, min=-100, max=100)
    
    labels = torch.arange(batch_size, device=q_reps.device)
    ce_loss = F.cross_entropy(scores, labels)
    
    l1_q = lambda_q * torch.mean(torch.abs(q_reps))
    l1_d = lambda_d * torch.mean(torch.abs(all_docs))
    
    total_loss = ce_loss + l1_q + l1_d
    
    if torch.isnan(total_loss) or torch.isinf(total_loss):
        return torch.tensor(0.0, device=q_reps.device, requires_grad=True)
    
    return total_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SPLADE().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)
scaler = torch.cuda.amp.GradScaler()

timings = {
    'data_to_device': 0.0,
    'forward_q': 0.0,
    'forward_p': 0.0,
    'forward_n': 0.0,
    'compute_loss': 0.0,
    'backward': 0.0,
    'optimizer_step': 0.0,
    'total_batch': 0.0
}

model.train()
batch_count = 0

for epoch in range(3):
    total_loss = 0.0
    running = 0.0
    valid_steps = 0

    for i, batch in enumerate(tqdm(dataloader, desc=f'Epoch {epoch+1}')):
        if batch_count >= 1000:
            break
            
        batch_start = time.time()
        
        optimizer.zero_grad(set_to_none=True)

        try:
            data_start = time.time()
            q_ids = batch['q_ids'].to(device)
            q_mask = batch['q_mask'].to(device)
            p_ids = batch['p_ids'].to(device)
            p_mask = batch['p_mask'].to(device)
            n_ids = batch['n_ids'].to(device)
            n_mask = batch['n_mask'].to(device)
            data_end = time.time()
            timings['data_to_device'] += data_end - data_start

            with torch.cuda.amp.autocast(dtype=torch.float16):
                forward_start = time.time()
                q_reps = model(q_ids, q_mask)
                forward_end = time.time()
                timings['forward_q'] += forward_end - forward_start

                forward_start = time.time()
                p_reps = model(p_ids, p_mask)
                forward_end = time.time()
                timings['forward_p'] += forward_end - forward_start

                forward_start = time.time()
                n_reps = model(n_ids, n_mask)
                forward_end = time.time()
                timings['forward_n'] += forward_end - forward_start

                loss_start = time.time()
                loss = compute_loss(q_reps, p_reps, n_reps)
                loss_end = time.time()
                timings['compute_loss'] += loss_end - loss_start

            if not torch.isnan(loss) and not torch.isinf(loss):
                backward_start = time.time()
                scaler.scale(loss).backward()
                backward_end = time.time()
                timings['backward'] += backward_end - backward_start

                step_start = time.time()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                step_end = time.time()
                timings['optimizer_step'] += step_end - step_start
                
                loss_val = loss.item()
                total_loss += loss_val
                running += loss_val
                valid_steps += 1
            else:
                scaler.update()
                continue

        except RuntimeError as e:
            scaler.update()
            continue

        batch_end = time.time()
        timings['total_batch'] += batch_end - batch_start
        batch_count += 1

        if batch_count >= 1000:
            break

    if batch_count >= 1000:
        break

print("\n=== Timing Results ===")
for operation, total_time in timings.items():
    avg_time = total_time / batch_count
    print(f"{operation}: {avg_time:.4f}s per batch (total: {total_time:.2f}s)")

print(f"\nTotal batches processed: {batch_count}")

/tmp/ipykernel_12167/825714394.py:49: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1:   0%|          | 0/1243151 [00:00<?, ?it/s]/tmp/ipykernel_12167/825714394.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
Epoch 1:   0%|          | 999/1243151 [01:56<40:21:03,  8.55it/s]



=== Timing Results ===
data_to_device: 0.0002s per batch (total: 0.22s)
forward_q: 0.0085s per batch (total: 8.52s)
forward_p: 0.0069s per batch (total: 6.87s)
forward_n: 0.0068s per batch (total: 6.85s)
compute_loss: 0.0030s per batch (total: 3.01s)
backward: 0.0351s per batch (total: 35.14s)
optimizer_step: 0.0057s per batch (total: 5.71s)
total_batch: 0.0693s per batch (total: 69.26s)

Total batches processed: 1000
